# Chapter 9: Learning Theory

> Simple functions generalize well — and now we can actually prove it, not just believe it.

**Type:** Learn + Build &nbsp;|&nbsp; **Language:** Python &nbsp;|&nbsp; **Prerequisites:** Chapter 1 (Decision Trees), Chapter 3 (The Perceptron) &nbsp;|&nbsp; **Time:** ~45 minutes
**Source:** *A Course in Machine Learning*, Hal Daumé III — Chapter 10

---

## Learning Objectives

- Explain why perfect, always-correct induction is provably impossible, and what "Probably Approximately Correct" (PAC) means instead
- Implement the "Throw Out Bad Terms" algorithm for PAC-learning boolean conjunctions and empirically verify its (ε, δ) behavior
- Demonstrate the VC dimension of linear classifiers in 2D by testing which point sets can and cannot be shattered
- Explain Occam's Razor as a formal sample-complexity statement, not just a philosophical preference for simplicity
- Recognize the difference between driving training error to zero and achieving low test error

## The Problem

Every algorithm in this course generalizes from a finite training sample to unseen data — but *why* should that ever work, and how many examples does it actually take?

This chapter turns "simple models generalize better" from an intuition into a theorem. There are two central ideas: (1) since a learner only ever sees a random sample, it can never be perfect every time — the best you can hope for is that it's *usually* *close to* correct (PAC learning); and (2) the size of your hypothesis class directly controls how many examples you need before you can trust that low training error implies low test error (Occam's Razor, VC dimension).

## The Concept

```
Hypothesis class H --> How large is H?
   Small (e.g. decision stump)     --> Few examples needed for train error to be trustworthy
   Large (e.g. unrestricted tree)  --> Many examples needed, or train error is misleading
   Both --> Occam's Bound: sample complexity grows with log|H|
```

### Key Ideas

- **Induction can't be perfect**: with label noise, or simply because your sample is finite and random, no algorithm can guarantee zero error on every possible training set. PAC learning formalizes the realistic goal instead: an (ε, δ)-PAC algorithm returns a function with error at most ε, with probability at least 1−δ
- **Occam's Razor as a theorem**: if a learner always fits the training data perfectly and its hypothesis class H is finite, the number of examples needed to guarantee low test error grows only with `log|H|` — not `|H|` itself. Simpler (smaller) hypothesis classes need dramatically fewer examples
- **VC dimension handles infinite hypothesis classes**: instead of counting hypotheses, you measure how many points the class can "shatter" (fit under every possible labeling). Linear classifiers in the plane can shatter any 3 points but never all labelings of 4 — so their VC dimension is exactly 3
- **Zero training error is not the goal**: an unbounded model (e.g., a fully-grown decision tree) can always reach 100% training accuracy, but that number tells you nothing about test performance. What matters is the gap between the two, and how quickly that gap closes as you add data

## Build It

### Setup

We'll need NumPy for array operations, and several utilities from scikit-learn: the Breast Cancer Wisconsin dataset (used as a real feature distribution throughout), a train/test splitter, `DecisionTreeClassifier` for Experiment C, `Perceptron` for the VC-dimension experiment, and an accuracy metric.

Real feature distributions come from this dataset throughout; only the *ground-truth labeling rule* in Experiment A is hand-specified, because PAC sample-complexity theorems are statements about learning a *known, fixed* target concept — you need to know the truth to measure "true" generalization error exactly, which no public dataset's real labels allow us to do.

In [1]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import Perceptron
from sklearn.metrics import accuracy_score

RNG = np.random.RandomState(0)

### Step 1: PAC-Learning a Boolean Conjunction — "Throw Out Bad Terms" (Algorithm 10.4)

The algorithm starts by assuming every possible literal (`x_d = 0` and `x_d = 1`, for every feature `d`) might be part of the true concept. Each time it sees a **positive** example, it throws out any literal that example violates — negative examples are ignored entirely. What survives at the end is exactly the conjunction of literals consistent with every positive example seen.

In [2]:
def binary_conjunction_train(X, y):
    D = X.shape[1]
    surviving = {(d, v) for d in range(D) for v in (0, 1)}
    for xi, yi in zip(X, y):
        if yi == 1:
            for d in range(D):
                literal_kept = (d, xi[d])
                literal_dropped = (d, 1 - xi[d])
                surviving.discard(literal_dropped)
    return surviving


def binary_conjunction_predict(surviving, X):
    preds = np.ones(X.shape[0], dtype=int)
    for (d, v) in surviving:
        preds &= (X[:, d] == v)
    return preds

### Experiment A: Empirically Verifying the (ε, δ)-PAC Guarantee

We fix a **known** ground-truth concept — a conjunction over 3 of the real (binarized) Breast Cancer features — so that "true" test error can be measured exactly against a large held-out pool. Then, for each training size `N`, we run 200 independent trials: sample `N` points, train the conjunction learner, and measure its error on the held-out evaluation set. As `N` grows, both the mean error and the fraction of "bad" runs (error above 5%) should collapse toward zero.

In [3]:
data = load_breast_cancer()
X_raw = data.data
medians = np.median(X_raw, axis=0)
X_bin_full = (X_raw > medians).astype(int)
D = X_bin_full.shape[1]
print(f"Using {X_bin_full.shape[0]} real (binarized) feature vectors, D={D} boolean features")

true_literals = {(2, 1), (5, 0), (7, 1)}
y_true_full = binary_conjunction_predict(true_literals, X_bin_full)
print(f"Ground-truth concept: literals {true_literals}")
print(f"Positive rate under this concept: {y_true_full.mean():.3f}")

X_pool, X_eval, y_pool, y_eval = train_test_split(
    X_bin_full, y_true_full, test_size=200, random_state=1
)

print(f"\n{'N (train size)':>15} | {'mean test err':>13} | {'P(err > 0.05)':>13}  (over 200 trials)")
print("-" * 60)
for N in [5, 10, 20, 40, 80, 160, 300]:
    errs = []
    for trial in range(200):
        rng = np.random.RandomState(trial)
        idx = rng.choice(len(X_pool), size=min(N, len(X_pool)), replace=False)
        Xs, ys = X_pool[idx], y_pool[idx]
        surviving = binary_conjunction_train(Xs, ys)
        preds = binary_conjunction_predict(surviving, X_eval)
        err = np.mean(preds != y_eval)
        errs.append(err)
    errs = np.array(errs)
    print(f"{N:>15} | {errs.mean():>13.4f} | {np.mean(errs > 0.05):>13.4f}")

Using 569 real (binarized) feature vectors, D=30 boolean features
Ground-truth concept: literals {(5, 0), (2, 1), (7, 1)}
Positive rate under this concept: 0.058

 N (train size) | mean test err | P(err > 0.05)  (over 200 trials)
------------------------------------------------------------
              5 |        0.0699 |        1.0000
             10 |        0.0696 |        1.0000
             20 |        0.0683 |        0.9750
             40 |        0.0651 |        0.8950


             80 |        0.0507 |        0.5650
            160 |        0.0195 |        0.0900
            300 |        0.0014 |        0.0000


**Reading the table:** as `N` grows from 5 to 300, both the mean test error and the probability of a "bad" run (error above 5%) shrink towards zero — exactly the (ε, δ)-PAC guarantee: with enough examples, the learned conjunction is *probably* (high `1 − δ`) *approximately* (low error `ε`) correct. Occam's bound for this hypothesis class (`|H| = 4^D`) gives a very loose worst-case sample complexity — in practice this simple algorithm converges far faster than that pessimistic bound suggests.

### Step 2: Testing Whether a Point Set Can Be Shattered (Section 10.6)

A set of points is **shattered** by a hypothesis class if, for *every* possible way of labeling them (every combination of positive/negative), some hypothesis in the class fits that labeling perfectly. For linear classifiers, we check this directly: try every binary labeling of the points, fit a perceptron, and see whether it recovers that exact labeling. If even one labeling defeats every possible line, the points are not shattered.

In [4]:
def can_shatter(points, max_iter=200):
    n = len(points)
    for labels_bits in range(2 ** n):
        y = np.array([(labels_bits >> i) & 1 for i in range(n)])
        if len(np.unique(y)) < 2:
            continue
        clf = Perceptron(max_iter=max_iter, tol=1e-3, random_state=0)
        clf.fit(points, y)
        preds = clf.predict(points)
        if not np.array_equal(preds, y):
            return False, y
    return True, None

### Experiment B: VC Dimension of Linear Classifiers in the Plane

The book's claim is that linear classifiers in 2D can shatter **any 3** points in general position (like a triangle), but **no set of 4** points — so their VC dimension is exactly 3. We test both directly: a triangle of 3 points, and a unit square of 4 points, where the diagonal-corners labeling is exactly the XOR pattern.

In [5]:
triangle = np.array([[0.0, 0.0], [1.0, 0.0], [0.0, 1.0]])
shattered_3, bad_labeling_3 = can_shatter(triangle)
print(f"Can a linear classifier shatter 3 points (triangle)?  {shattered_3}")

square = np.array([[0.0, 0.0], [1.0, 0.0], [0.0, 1.0], [1.0, 1.0]])
shattered_4, bad_labeling_4 = can_shatter(square)
print(f"Can a linear classifier shatter 4 points (unit square)? {shattered_4}")
if not shattered_4:
    print(f"  Counter-example labeling that fails: {bad_labeling_4}  "
          f"(this is the XOR pattern: opposite corners share a label)")

Can a linear classifier shatter 3 points (triangle)?  True
Can a linear classifier shatter 4 points (unit square)? False
  Counter-example labeling that fails: [0 1 1 0]  (this is the XOR pattern: opposite corners share a label)


**Reading the result:** this matches the theorem in Section 10.6: the VC dimension of linear classifiers in 2D is exactly 3 — some set of 3 points can always be shattered, but every set of 4 points has at least one labeling (XOR) that no line can separate.

### Experiment C: Occam's Razor in Practice — Decision Stump vs. Unrestricted Tree

Here we contrast a **small** hypothesis class (a decision stump, `max_depth=1`) against a **large** one (an unrestricted decision tree, `max_depth=None`), training both on increasingly large subsets of the real Breast Cancer dataset and evaluating on a fixed held-out test set. Occam's Razor predicts that the smaller class should need far fewer examples before its training accuracy becomes a trustworthy estimate of its test accuracy.

In [6]:
y = np.where(data.target == 0, 0, 1)
X = data.data
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)
print(f"Full training pool: {len(X_train_full)}  |  fixed test set: {len(X_test)}")

print(f"\n{'N':>5} | {'stump train':>11} | {'stump test':>10} | {'full-tree train':>16} | {'full-tree test':>15}")
print("-" * 72)
for N in [10, 20, 40, 80, 160, 300, len(X_train_full)]:
    rng = np.random.RandomState(0)
    idx = rng.choice(len(X_train_full), size=N, replace=False)
    Xs, ys = X_train_full[idx], y_train_full[idx]

    stump = DecisionTreeClassifier(max_depth=1, random_state=0).fit(Xs, ys)
    full_tree = DecisionTreeClassifier(max_depth=None, random_state=0).fit(Xs, ys)

    stump_train = accuracy_score(ys, stump.predict(Xs))
    stump_test = accuracy_score(y_test, stump.predict(X_test))
    tree_train = accuracy_score(ys, full_tree.predict(Xs))
    tree_test = accuracy_score(y_test, full_tree.predict(X_test))
    print(f"{N:>5} | {stump_train:>11.4f} | {stump_test:>10.4f} | {tree_train:>16.4f} | {tree_test:>15.4f}")

Full training pool: 398  |  fixed test set: 171

    N | stump train | stump test |  full-tree train |  full-tree test
------------------------------------------------------------------------
   10 |      1.0000 |     0.6901 |           1.0000 |          0.6901
   20 |      1.0000 |     0.8655 |           1.0000 |          0.8655
   40 |      1.0000 |     0.8596 |           1.0000 |          0.8596
   80 |      0.9625 |     0.9123 |           1.0000 |          0.9240
  160 |      0.9625 |     0.9064 |           1.0000 |          0.9064
  300 |      0.9467 |     0.9064 |           1.0000 |          0.9123
  398 |      0.9271 |     0.9123 |           1.0000 |          0.9181


**Reading the table:** the unrestricted tree always reaches close to 100% training accuracy, at every `N` (it can memorize any training set, however small or large) — and its train/test gap never closes. The stump's training accuracy actually *drops* as `N` grows (it runs out of room to fit more points with just one split), and its train and test numbers stay close together throughout.

This is Occam's point in miniature: a huge hypothesis class can always drive training error to zero, but that zero is not informative about test performance — the persistent gap for the full tree is the symptom of a hypothesis class large enough to memorize whatever it sees, regardless of how many examples you give it.

## When to Use These Ideas

| API / Function | When to use it |
|---|---|
| `binary_conjunction_train(X, y)` | Illustrative only — real problems rarely have a noise-free boolean conjunction as their true concept, but the algorithm is the cleanest possible illustration of PAC convergence |
| `can_shatter(points)` | A didactic tool for building intuition about VC dimension; not something you'd run on production data |
| Learning curves (train/test accuracy vs. `N`) | The practical, everyday tool this chapter justifies theoretically — always plot this before trusting a complex model on limited data |
| `max_depth` / other capacity hyperparameters | The practical lever for controlling `|H|`, directly connecting back to Occam's Bound |

## Exercises

1. Re-run Experiment A with a *noisy* ground-truth concept (flip 5% of labels at random) and observe how the PAC guarantee degrades — does the algorithm ever converge to zero error?
2. Compute the actual VC dimension of axis-aligned rectangles in 2D by extending `can_shatter` — how many points can be shattered?
3. In Experiment C, add irrelevant random noise features to the dataset and re-run the learning curves — does the gap between the stump and the full tree get worse, matching the "irrelevant features" discussion from Chapter 4?

## Key Terms

| Term | Common Assumption | Precise Meaning |
|---|---|---|
| **PAC Learning** | "The algorithm always works" | A guarantee that an algorithm returns a low-error (ε) hypothesis with high probability (1−δ), not that it is always low-error |
| **Occam's Razor** | "Prefer simple explanations, philosophically" | A formal theorem: for a finite hypothesis class that fits training data perfectly, sample complexity scales with `log|H|` |
| **VC Dimension** | "How complex a model looks" | The largest number of points a hypothesis class can shatter — i.e., fit under every possible labeling — used to measure the capacity of infinite hypothesis classes |
| **Sample Complexity** | "How much data you happen to have" | The number of training examples an algorithm formally requires to guarantee a target error rate with a target confidence |

## Summary

- **PAC learning** replaces the impossible goal of "always perfectly correct" with a realistic one: probably (`1 − δ`) approximately (`ε`) correct
- **Occam's Razor**, made precise, says sample complexity for a finite hypothesis class scales with `log|H|`, not `|H|` — simplicity is a quantitative advantage, not just an aesthetic preference
- **VC dimension** extends this idea to infinite hypothesis classes by counting how many points a class can shatter — linear classifiers in 2D have VC dimension exactly 3
- **Zero training error means nothing on its own** — an unrestricted decision tree can always memorize its training set, but the persistent train/test gap is exactly the symptom a small hypothesis class avoids
- All three experiments reproduce these theorems empirically, on real feature distributions, rather than leaving them as diagrams

---

**Next:** Chapter 10 — beyond learning theory